# 13 - Land-cover diagnostic: is the recovery-gap ordering in the target, in fire timing, or in the model?

**Why this notebook exists.** The polygon / spatial-unit / land-cover aggregation work (Notebooks 11-12, Options 1/2/3) and the GEE export are paused. Before deciding whether aggregating indicators by land cover is worth doing, the underlying scientific question has to be answered: *are the recovery-gap differences between land-cover classes real ecological recovery differences, or are they already baked into the NBR-derived target and/or produced by when each class happened to burn?*

**Constraints.**
- Stored arrays only. No model is retrained, no Earth Engine call is made, no stored target is modified.
- Burned pixels only: a pixel is kept if it has at least one labelled fire month in `y_burned.npy`, i.e. it intersects the fire-reference inventory inside the Jan 2017 - Dec 2025 label window. This is also the population the regression heads were trained on.
- Everything this notebook writes goes to `diagnostics/nb13_landcover_timing/` and is prefixed `nb13_`. **Nothing is written into `outputs/`** - a cell below asserts it.

**Definitions used throughout** (all verified in §1 against the stored arrays, not assumed):

| quantity | definition | array |
|---|---|---|
| recovery gap (target) | `clip((b - r) / (abs(b) + 1e-6), 0, 1)`, `b` = mean NBR Jan-Dec 2017 (months 0-11), `r` = mean NBR Jan-Dec 2025 (months 96-107). **Normalised by the pixel's own 2017 level**, not a plain difference. | `targets_aux.npz["recovery_gap"]`, rebuilt from `nbr_raw.npy` |
| recurrence (target) | `min(fire-month count / 5, 1)`. A pixel with one fire has recurrence **0.2**; "recurrence = 1 fire" below means count = 1. | `targets_aux.npz["recurrence"]`, `y_burned.npy` |
| first fire month | first month with a fire label, index 0 = Jan 2017 | `y_burned.npy` |
| land cover | `lc_dominant`, the modal current-year class across the 108 months | `X_scaled_meta.csv` |
| valid observation | `True` where the monthly composite was observed; `False` months hold imputed values | `X_scaled_mask.npy` |
| model outputs | test-split predictions of Notebook 05; model recurrence rebuilt with the Notebook 07b rule | `models/main/predictions_test.npz` |

## Contents

| § | Question | Main outputs |
|---|---|---|
| 1 | Inputs, fingerprints, and verification that the arrays reproduce the stored targets | t00, t00b |
| 2 | Fire timing per class - when did each class first burn? | t01a-c, f01 |
| 3 | Event-centred recovery curve, single-fire pixels, -12 to +24 months | t02a-b, f02 |
| 4 | Targets vs model outputs per class | t03a-b, f03 |
| 5 | Recovery-gap target distribution per class | t04a-b, f04 |
| 6 | Fixed-cohort control - first fire in 2018-2019 | t05a-c, f05 |
| 7 | Baseline (2016-2017) seasonality per class | t06a-c, f06 |
| 8 | Verdict | nb13_verdict.md |
| 9 | Result registry | t99 |

In [ ]:
import hashlib
import json
import os
import platform
import sys
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

# Resolve the project root whether the notebook is run from notebooks/ or the root.
ROOT = os.environ.get("THESIS_ROOT")
if not ROOT:
    here = os.path.abspath(os.getcwd())
    ROOT = os.path.dirname(here) if os.path.basename(here) == "notebooks" else here
ROOT = os.path.abspath(os.path.expanduser(ROOT))

PROC = os.path.join(ROOT, "data", "processed")
MAIN = os.path.join(ROOT, "models", "main")
OUT_REL = os.path.join("diagnostics", "nb13_landcover_timing")
OUT = os.path.join(ROOT, OUT_REL)
TAB = os.path.join(OUT, "tables")
FIG = os.path.join(OUT, "figures")

# Hard guard: this diagnostic must not write anything into outputs/.
_outputs = os.path.join(ROOT, "outputs")
assert os.path.commonpath([OUT, _outputs]) != _outputs, "refusing to write under outputs/"
os.makedirs(TAB, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
print("thesis root :", ROOT)
print("writes to   :", OUT)
print("run (UTC)   :", RUN_STAMP)
print("python      :", sys.version.split()[0], "|", platform.platform())
print("numpy", np.__version__, "| pandas", pd.__version__,
      "| scipy", __import__("scipy").__version__, "| matplotlib", matplotlib.__version__)

In [ ]:
# ---- shared constants and helpers -----------------------------------------
LC_NAMES = {10: "forest", 11: "orchards", 20: "shrubland", 30: "grassland",
            40: "cropland", 41: "covered agriculture", 50: "built-up",
            60: "bare", 61: "fallow", 80: "water"}
LC_ORDER = [10, 11, 20, 30, 40, 41, 50, 60, 61, 80]
_cmap = plt.get_cmap("tab10")
COLOR = {c: _cmap(i) for i, c in enumerate(LC_ORDER)}

MIN_N = 500          # classes below this are computed but flagged as small-n
EPS = 1e-6           # the epsilon in the recovery-gap denominator (Notebook 04)
N_MONTHS = 108       # Jan 2017 - Dec 2025
YEARS = list(range(2017, 2026))
IND = ["severity", "persistence", "recovery_gap", "recurrence"]

PREFIX = "nb13_"
REGISTRY = []

matplotlib.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9,
                            "axes.grid": True, "grid.alpha": 0.25,
                            "axes.spines.top": False, "axes.spines.right": False})


def _fingerprint(path, edge=16 * 1024 * 1024):
    """SHA-256 over the first and last `edge` bytes (same scheme as Notebook 08)."""
    h = hashlib.sha256()
    size = os.path.getsize(path)
    with open(path, "rb") as fh:
        h.update(fh.read(min(edge, size)))
        if size > edge:
            fh.seek(max(0, size - edge))
            h.update(fh.read(edge))
    return h.hexdigest()[:16], size


def _markdown_table(df, fmt="{:.4f}"):
    def cell(v):
        if isinstance(v, (bool, np.bool_)):
            return str(bool(v))
        if isinstance(v, (float, np.floating)):
            return "" if not np.isfinite(v) else fmt.format(v)
        return str(v)
    cols = list(df.columns)
    out = ["| " + " | ".join(str(c) for c in cols) + " |",
           "|" + "|".join("---" for _ in cols) + "|"]
    for _, row in df.iterrows():
        out.append("| " + " | ".join(cell(row[c]) for c in cols) + " |")
    return "\n".join(out) + "\n"


def _sources_md(sources):
    return "\n".join("- `{}` — {}".format(k, v) for k, v in sources.items())


def save_table(df, stem, caption, section, sources):
    """CSV + Markdown; the Markdown carries a per-column source-array list."""
    name = PREFIX + stem
    df.to_csv(os.path.join(TAB, name + ".csv"), index=False)
    with open(os.path.join(TAB, name + ".md"), "w") as fh:
        fh.write("<!-- generated by notebooks/13_landcover_timing_diagnostic.ipynb §{} on {} -->\n\n"
                 .format(section, RUN_STAMP))
        fh.write(caption.rstrip() + "\n\n**Sources (array → column).**\n\n" + _sources_md(sources) + "\n\n")
        fh.write(_markdown_table(df))
    REGISTRY.append(dict(section=str(section), kind="table", stem=name,
                         files=os.path.join(OUT_REL, "tables", name) + ".csv, .md",
                         caption=caption, sources="; ".join(sorted(set(sources.values())))))
    print("  ->", os.path.join(OUT_REL, "tables", name + ".csv"), "(+ .md)")
    return df


def save_figure(fig, stem, caption, section, sources):
    name = PREFIX + stem
    path = os.path.join(FIG, name + ".png")
    fig.savefig(path, bbox_inches="tight")
    with open(os.path.join(FIG, name + ".md"), "w") as fh:
        fh.write("<!-- generated by notebooks/13_landcover_timing_diagnostic.ipynb §{} on {} -->\n\n"
                 .format(section, RUN_STAMP))
        fh.write(caption.rstrip() + "\n\n**Sources.**\n\n" + _sources_md(sources) + "\n")
    REGISTRY.append(dict(section=str(section), kind="figure", stem=name,
                         files=os.path.join(OUT_REL, "figures", name) + ".png, .md",
                         caption=caption, sources="; ".join(sorted(set(sources.values())))))
    print("  ->", os.path.join(OUT_REL, "figures", name + ".png"))
    return path


def note(*msg):
    print("   ", *msg)

## 1. Inputs, fingerprints and verification

**Question.** Are the stored arrays the ones the targets were built from, and do the definitions in the header hold?

**Why.** Every number below is computed from these arrays. `nbr_raw.npy` and `nbr_zscore_raw.npy` are derived files (extracted from `X_scaled.npy` by `notebooks/extract_nbr_arrays.py`, see Notebook 08 §1), so this notebook verifies them itself rather than trusting that script or Notebook 08.

**Data.** `targets_aux.npz`, `X_scaled_meta.csv`, `y_burned.npy`, `X_scaled_mask.npy`, `nbr_raw.npy`, `nbr_zscore_raw.npy`, `splits.json`, `predictions_test.npz`.

**Metric.** (i) the burned set from `y_burned` equals `recurrence > 0`; (ii) the recovery gap rebuilt from `nbr_raw` with the Notebook 04 formula matches the stored target (Pearson > 0.9999 and > 99.9% of pixels within 1e-2 - the Notebook 08 criterion, because inverting float32 standardisation costs a few ULPs that the small denominator amplifies); (iii) recurrence equals `min(count/5, 1)`; (iv) the fire labels stored with the test predictions equal `y_burned` for those pixels.

**Reading the output.** Every check must print `True`; the cell asserts it.

In [ ]:
INPUTS = {
    "targets_aux.npz":      os.path.join(PROC, "targets_aux.npz"),
    "X_scaled_meta.csv":    os.path.join(PROC, "X_scaled_meta.csv"),
    "y_burned.npy":         os.path.join(PROC, "y_burned.npy"),
    "X_scaled_mask.npy":    os.path.join(PROC, "X_scaled_mask.npy"),
    "nbr_raw.npy":          os.path.join(PROC, "nbr_raw.npy"),
    "nbr_zscore_raw.npy":   os.path.join(PROC, "nbr_zscore_raw.npy"),
    "splits.json":          os.path.join(PROC, "splits.json"),
    "predictions_test.npz": os.path.join(MAIN, "predictions_test.npz"),
}
rows = []
for label, path in INPUTS.items():
    assert os.path.exists(path), "missing input: " + path
    fp, size = _fingerprint(path)
    rows.append(dict(input=label, path=os.path.relpath(path, ROOT),
                     size_MB=round(size / 1e6, 1), sha256_head_tail=fp))
inventory = pd.DataFrame(rows)
display(inventory)
save_table(inventory, "t00_input_inventory",
           "Inputs consumed by Notebook 13, with a head+tail SHA-256 fingerprint "
           "(first and last 16 MiB).", section=1,
           sources={"sha256_head_tail": "file bytes of each input"})

In [ ]:
targets = dict(np.load(INPUTS["targets_aux.npz"]))
meta = pd.read_csv(INPUTS["X_scaled_meta.csv"])
y_all = np.load(INPUTS["y_burned.npy"], mmap_mode="r")
N_PIX = len(meta)
assert y_all.shape == (N_PIX, N_MONTHS)
LC_ALL = meta["lc_dominant"].to_numpy()

BURNED = np.asarray(y_all > 0).any(axis=1)
BIDX = np.flatnonzero(BURNED)
N_BURNED = len(BIDX)

yb = np.asarray(y_all[BIDX]) > 0
lcb = LC_ALL[BIDX]
nbr_b = np.asarray(np.load(INPUTS["nbr_raw.npy"], mmap_mode="r")[BIDX], dtype=np.float32)
nbz_b = np.asarray(np.load(INPUTS["nbr_zscore_raw.npy"], mmap_mode="r")[BIDX], dtype=np.float32)
valid_b = np.asarray(np.load(INPUTS["X_scaled_mask.npy"], mmap_mode="r")[BIDX], dtype=bool)
TG = {k: targets[k][BIDX].astype(np.float64) for k in IND}

n_fire = yb.sum(axis=1)                                   # labelled fire months
first = np.argmax(yb, axis=1)                             # first fire month index
last = N_MONTHS - 1 - np.argmax(yb[:, ::-1], axis=1)      # last fire month index
ffy = 2017 + first // 12                                  # first-fire calendar year
ffdate = 2017 + first / 12.0                              # first-fire date, decimal years

# ---- verification ---------------------------------------------------------
b = nbr_b[:, :12].astype(np.float64).mean(axis=1)
r = nbr_b[:, 96:108].astype(np.float64).mean(axis=1)
rg_chk = np.clip((b - r) / (np.abs(b) + EPS), 0, 1)
rg_diff = np.abs(rg_chk - TG["recovery_gap"])

preds = np.load(INPUTS["predictions_test.npz"])
TIDX = preds["indices"]
splits = json.load(open(INPUTS["splits.json"]))
labels_test = preds["burned_labels"].reshape(len(TIDX), N_MONTHS) > 0

checks = pd.DataFrame([
    dict(check="burned set from y_burned == (recurrence target > 0)",
         value=str(int(N_BURNED)) + " pixels",
         passed=bool(np.array_equal(BURNED, targets["recurrence"] > 0))),
    dict(check="recovery_gap rebuilt from nbr_raw: Pearson vs stored",
         value="{:.8f}".format(np.corrcoef(rg_chk, TG["recovery_gap"])[0, 1]),
         passed=bool(np.corrcoef(rg_chk, TG["recovery_gap"])[0, 1] > 0.9999)),
    dict(check="recovery_gap rebuilt from nbr_raw: share within 1e-2",
         value="{:.6f}".format((rg_diff < 1e-2).mean()),
         passed=bool((rg_diff < 1e-2).mean() > 0.999)),
    dict(check="recurrence target == min(fire-month count / 5, 1)",
         value="max |diff| {:.1e}".format(np.abs(np.minimum(n_fire / 5, 1) - TG["recurrence"]).max()),
         passed=bool(np.allclose(np.minimum(n_fire / 5, 1), TG["recurrence"], atol=1e-6))),
    dict(check="predictions_test indices == splits.json test split",
         value=str(len(TIDX)) + " pixels",
         passed=bool(np.array_equal(np.sort(TIDX), np.sort(np.asarray(splits["test"]))))),
    dict(check="fire labels stored with predictions == y_burned[test]",
         value="", passed=bool(np.array_equal(labels_test, np.asarray(y_all[TIDX]) > 0))),
])
display(checks)
assert checks["passed"].all(), "verification failed — stop here"
note("burned pixels:", format(N_BURNED, ","), "of", format(N_PIX, ","))
note("pixels typed low_recurrence but with no labelled fire month (excluded):",
     format(int(((meta["pixel_type"] != "no_fires").to_numpy() & ~BURNED).sum()), ","))
note("imputed (invalid) observations in the 2017 window: {:.2f}%, in the 2025 window: {:.2f}%"
     .format(100 * (1 - valid_b[:, :12].mean()), 100 * (1 - valid_b[:, 96:].mean())))
save_table(checks, "t00b_verification",
           "Verification that the stored arrays reproduce the stored targets and the "
           "definitions used in this notebook. All checks must pass.", section=1,
           sources={"recovery_gap rebuilt": "nbr_raw.npy[burned, 0:12] and [burned, 96:108] vs targets_aux.npz['recovery_gap']",
                    "burned set": "y_burned.npy vs targets_aux.npz['recurrence']",
                    "recurrence": "y_burned.npy (count of labelled months) vs targets_aux.npz['recurrence']",
                    "test split": "predictions_test.npz['indices'], splits.json['test'], predictions_test.npz['burned_labels'], y_burned.npy"})

## 2. Fire timing per class

**Question.** When did each land-cover class first burn, and do cropland and forest differ?

**Why.** The recovery gap compares fixed calendar windows (2017 vs 2025). A class whose pixels burned earlier has had more time to recover by 2025 and should show a *smaller* gap for that reason alone. If classes differ systematically in fire timing, that is a candidate explanation for the ordering, independent of ecology.

**Data.** `y_burned.npy` (first and last labelled fire month), `X_scaled_meta.csv` (`lc_dominant`). Burned pixels only.

**Metric.** Count and share of burned pixels by first-fire year per class; median and IQR of the first-fire date; share in 2017 (inside the reference window), 2018-2019, 2020-2022, 2023-2025; mean months between the first / last fire and the start of the 2025 window (month 96), over pixels whose fire precedes it. `n_distinct_first_fire_months` counts how many different calendar months the first fires fall in - pixels burned in the same month are largely the same fire events, so this is a reminder that pixel n overstates the number of independent events. For cropland vs forest: difference in medians and shares, and the two-sample Kolmogorov-Smirnov statistic D as an effect size (p-values are omitted: with tens of thousands of spatially autocorrelated pixels they carry no information).

**Reading the output.** A timing artefact would require the class with the *lower* recovery gap to have burned *earlier*.

In [ ]:
rows_n, rows_s = [], []
for c in LC_ORDER:
    m = lcb == c
    n = int(m.sum())
    if n == 0:
        continue
    rn = dict(land_cover=LC_NAMES[c], n=n)
    for yr in YEARS:
        rn["n_" + str(yr)] = int((m & (ffy == yr)).sum())
    for yr in YEARS:
        rn["pct_" + str(yr)] = 100.0 * rn["n_" + str(yr)] / n
    rows_n.append(rn)

    f, lf, d = first[m], last[m], ffdate[m]
    rows_s.append(dict(
        land_cover=LC_NAMES[c], n=n, small_n=n < MIN_N,
        n_distinct_first_fire_months=int(len(np.unique(f))),
        median_first_fire=float(np.median(d)),
        p25_first_fire=float(np.percentile(d, 25)), p75_first_fire=float(np.percentile(d, 75)),
        mean_first_fire=float(d.mean()),
        pct_2017=100 * float((ffy[m] == 2017).mean()),
        pct_2018_2019=100 * float(np.isin(ffy[m], [2018, 2019]).mean()),
        pct_2020_2022=100 * float(np.isin(ffy[m], [2020, 2021, 2022]).mean()),
        pct_2023_2025=100 * float(np.isin(ffy[m], [2023, 2024, 2025]).mean()),
        mean_months_first_fire_to_2025_window=float((96 - f[f < 96]).mean()),
        mean_months_last_fire_to_2025_window=float((96 - lf[lf < 96]).mean()),
        pct_single_fire=100 * float((n_fire[m] == 1).mean()),
        recovery_gap_target_mean=float(TG["recovery_gap"][m].mean())))
timing_counts = pd.DataFrame(rows_n)
timing = pd.DataFrame(rows_s).sort_values("median_first_fire")
display(timing_counts)
display(timing)

fo, cr = lcb == 10, lcb == 40
ks = stats.ks_2samp(ffdate[cr], ffdate[fo])
cmp_rows = []
for metric, fn in [("n burned pixels", lambda m: float(m.sum())),
                   ("median first-fire date (decimal year)", lambda m: float(np.median(ffdate[m]))),
                   ("mean first-fire date (decimal year)", lambda m: float(ffdate[m].mean())),
                   ("% first fire in 2017", lambda m: 100 * float((ffy[m] == 2017).mean())),
                   ("% first fire in 2018-2019", lambda m: 100 * float(np.isin(ffy[m], [2018, 2019]).mean())),
                   ("% first fire in 2023-2025", lambda m: 100 * float(np.isin(ffy[m], [2023, 2024, 2025]).mean())),
                   ("mean months, first fire -> Jan 2025", lambda m: float((96 - first[m & (first < 96)]).mean())),
                   ("mean months, last fire -> Jan 2025", lambda m: float((96 - last[m & (last < 96)]).mean())),
                   ("recovery-gap target mean", lambda m: float(TG["recovery_gap"][m].mean()))]:
    a, bb = fn(fo), fn(cr)
    cmp_rows.append(dict(metric=metric, forest=a, cropland=bb, cropland_minus_forest=bb - a))
cmp_rows.append(dict(metric="KS statistic D (first-fire date distributions)", forest=np.nan,
                     cropland=np.nan, cropland_minus_forest=float(ks.statistic)))
timing_cf = pd.DataFrame(cmp_rows)
display(timing_cf)

SRC_T = {"first-fire month / year": "y_burned.npy (argmax of labelled months), burned pixels",
         "last-fire month": "y_burned.npy (last labelled month)",
         "land_cover": "X_scaled_meta.csv['lc_dominant']",
         "pct_single_fire": "y_burned.npy (labelled-month count == 1)",
         "recovery_gap_target_mean": "targets_aux.npz['recovery_gap']"}
save_table(timing_counts, "t01a_first_fire_year_counts",
           "Burned pixels by land-cover class and year of first labelled fire: counts and "
           "row percentages.", section=2, sources=SRC_T)
save_table(timing, "t01b_fire_timing_summary",
           "Fire timing by land-cover class, sorted by median first-fire date. 'mean_months_*' "
           "are measured to the start of the 2025 assessment window (Jan 2025) over pixels whose "
           "fire precedes it.", section=2, sources=SRC_T)
save_table(timing_cf, "t01c_cropland_vs_forest_timing",
           "Cropland versus forest fire timing. KS D is reported as an effect size only.",
           section=2, sources=SRC_T)

In [ ]:
show = timing[~timing["small_n"]]["land_cover"].tolist()
fig, ax = plt.subplots(figsize=(8.5, 0.45 * len(timing) + 1.2))
ycols = plt.get_cmap("viridis")(np.linspace(0.05, 0.95, len(YEARS)))
order = timing["land_cover"].tolist()[::-1]
tc = timing_counts.set_index("land_cover")
left = np.zeros(len(order))
for j, yr in enumerate(YEARS):
    vals = tc.loc[order, "pct_" + str(yr)].to_numpy()
    ax.barh(range(len(order)), vals, left=left, color=ycols[j], label=str(yr), height=0.75)
    left += vals
ax.set_yticks(range(len(order)))
ax.set_yticklabels(["{} (n={:,}){}".format(k, int(tc.loc[k, "n"]), " *" if tc.loc[k, "n"] < MIN_N else "")
                    for k in order])
ax.set_xlabel("% of burned pixels in class")
ax.set_xlim(0, 100)
ax.legend(title="first fire", ncol=9, fontsize=7, title_fontsize=8,
          loc="upper center", bbox_to_anchor=(0.45, -0.12), frameon=False)
ax.set_title("Year of first labelled fire by land-cover class (sorted by median first-fire date)\n"
             "* n < {} — small-n".format(MIN_N), fontsize=10)
fig.tight_layout()
save_figure(fig, "f01_first_fire_year_by_class",
            "Share of burned pixels by year of first labelled fire, per land-cover class, "
            "sorted by median first-fire date (earliest at the top).", section=2, sources=SRC_T)
plt.show()

## 3. Event-centred recovery curve - single-fire pixels

**Question.** Measured directly on each pixel's own fire rather than through the fixed-window indicator, what does post-fire NBR do in each class?

**Why.** Aligning on the fire month removes the fire-timing problem by construction: every pixel contributes the same -12 to +24 month window around its own fire. Restricting to single-fire pixels removes the confound of a second fire resetting recovery inside the window.

**Data.** `nbr_raw.npy`, `X_scaled_mask.npy`, `y_burned.npy`, `X_scaled_meta.csv`. Burned pixels with exactly one labelled fire month (recurrence target 0.2) whose fire month leaves 12 months before and 24 after inside the record (fire between Jan 2018 and Dec 2023). Only valid observations are averaged; imputed months are excluded.

**Metric.** For each pixel, `pre` = mean NBR over months -12..-1. Curves: mean NBR and mean (NBR - pre) at each offset. Summary: ΔNBR at the fire month, trough ΔNBR and its offset, mean ΔNBR over year 1 (+1..+12) and year 2 (+13..+24) - whole 12-month blocks, so the annual cycle cancels in these two numbers - and the share of pixels whose year-2 mean is still below `pre`. The recovery-gap target mean of the same pixels is shown alongside. Because classes burned in different years (§2), the year-2 numbers are repeated for fires in 2018-2019 only, so class is not confounded with calendar period (e.g. forest's fires are mostly October 2023, so its year 2 falls in 2024-2025).

**Reading the output.** The monthly curves still carry the annual cycle (fires cluster in the dry season), so compare classes on the block means. Classes with n < 500 are listed but not plotted.

In [ ]:
OFFS = np.arange(-12, 25)          # 37 offsets; column j <-> offset j - 12
J0 = 12                            # column of the fire month
single = n_fire == 1
ev_ok = single & (first >= 12) & (first + 24 <= N_MONTHS - 1)
note("single-fire burned pixels: {:,}; with a complete -12/+24 window: {:,}"
     .format(int(single.sum()), int(ev_ok.sum())))

curve_rows, ev_rows, CURVES = [], [], {}
for c in LC_ORDER:
    m = ev_ok & (lcb == c)
    n = int(m.sum())
    if n == 0:
        continue
    f = first[m]
    ii = f[:, None] + OFFS[None, :]
    rr = np.arange(n)[:, None]
    v = valid_b[m][rr, ii]
    w = np.where(v, nbr_b[m][rr, ii].astype(np.float64), np.nan)
    pre = np.nanmean(w[:, :J0], axis=1)
    d = w - pre[:, None]
    mean_abs = np.nanmean(w, axis=0)
    mean_d = np.nanmean(d, axis=0)
    y1 = np.nanmean(w[:, J0 + 1:J0 + 13], axis=1)
    y2 = np.nanmean(w[:, J0 + 13:J0 + 25], axis=1)
    CURVES[c] = (n, mean_abs, mean_d)
    for j, o in enumerate(OFFS):
        curve_rows.append(dict(land_cover=LC_NAMES[c], offset_month=int(o), n_pixels=n,
                               n_valid_obs=int(v[:, j].sum()), mean_NBR=float(mean_abs[j]),
                               mean_dNBR_vs_prefire=float(mean_d[j])))
    ok2 = np.isfinite(pre) & np.isfinite(y2)
    rel = ok2 & (pre > 0.01)
    post = mean_d[J0:]
    early = np.isin(ffy[m], [2018, 2019])        # same calendar period for every class
    okc = ok2 & early
    ev_rows.append(dict(
        land_cover=LC_NAMES[c], n=n, small_n=n < MIN_N,
        median_fire_date=float(np.median(ffdate[m])),
        prefire_NBR_mean=float(np.nanmean(pre)),
        dNBR_fire_month=float(mean_d[J0]),
        trough_dNBR=float(np.nanmin(post)),
        trough_offset=int(OFFS[J0 + int(np.nanargmin(post))]),
        dNBR_year1_mean=float(np.nanmean(y1 - pre)),
        dNBR_year2_mean=float(np.nanmean(y2 - pre)),
        rel_dNBR_year2_mean=float(((y2[rel] - pre[rel]) / pre[rel]).mean()),
        n_rel=int(rel.sum()),
        pct_pixels_year2_below_prefire=100 * float((y2[ok2] < pre[ok2]).mean()),
        n_fires_2018_2019=int(early.sum()),
        dNBR_year2_mean_fires_2018_2019=float(np.nanmean((y2 - pre)[early])) if early.any() else np.nan,
        pct_year2_below_prefire_fires_2018_2019=100 * float((y2[okc] < pre[okc]).mean()) if okc.any() else np.nan,
        recovery_gap_target_mean=float(TG["recovery_gap"][m].mean())))
event_curves = pd.DataFrame(curve_rows)
event_summary = pd.DataFrame(ev_rows).sort_values("dNBR_year2_mean")
display(event_summary)

SRC_E = {"NBR values": "nbr_raw.npy (valid months only, X_scaled_mask.npy == True)",
         "fire month / single-fire selection": "y_burned.npy (labelled-month count == 1)",
         "land_cover": "X_scaled_meta.csv['lc_dominant']",
         "recovery_gap_target_mean": "targets_aux.npz['recovery_gap']"}
save_table(event_curves, "t02a_event_centred_curves",
           "Mean NBR and mean NBR change from the pre-fire year (months -12..-1) at each "
           "month offset from the fire, single-fire burned pixels with a complete -12/+24 "
           "window, by land-cover class. These are the plotted values of f02.",
           section=3, sources=SRC_E)
save_table(event_summary, "t02b_event_centred_summary",
           "Event-centred post-fire NBR response by land-cover class, single-fire pixels. "
           "'dNBR_year1/2_mean' are whole-year block means minus the pre-fire year, so the "
           "annual cycle cancels; 'rel_dNBR_year2_mean' divides by the pre-fire level where "
           "it exceeds 0.01 (n_rel pixels). Negative = below pre-fire. The '*_fires_2018_2019' "
           "columns repeat year 2 for fires in 2018-2019 only, so every class is compared over "
           "the same calendar period.", section=3, sources=SRC_E)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharex=True)
for c, (n, mean_abs, mean_d) in CURVES.items():
    if n < MIN_N:
        continue
    lab = "{} (n={:,})".format(LC_NAMES[c], n)
    axes[0].plot(OFFS, mean_abs, color=COLOR[c], lw=1.6, label=lab)
    axes[1].plot(OFFS, mean_d, color=COLOR[c], lw=1.6, label=lab)
for ax in axes:
    ax.axvline(0, color="crimson", ls="--", lw=1)
    ax.set_xticks(np.arange(-12, 25, 6))
    ax.set_xlabel("months relative to the (single) labelled fire")
axes[1].axhline(0, color="k", lw=0.8)
axes[0].set_ylabel("mean NBR")
axes[1].set_ylabel("mean NBR − pixel's pre-fire-year mean")
axes[0].set_title("Absolute NBR", fontsize=10)
axes[1].set_title("Change from the 12 months before the fire", fontsize=10)
axes[0].legend(fontsize=7.5, loc="upper right")
fig.suptitle("Event-centred NBR trajectory by land cover — single-fire burned pixels, fires Jan 2018–Dec 2023",
             fontsize=11)
fig.tight_layout()
save_figure(fig, "f02_event_centred_nbr_single_fire",
            "Mean NBR aligned on each pixel's single labelled fire (month 0), -12 to +24 months, "
            "by land-cover class (n per class in legend; classes with n < 500 omitted). Valid "
            "observations only. Monthly values retain the annual cycle.", section=3, sources=SRC_E)
plt.show()

## 4. Analytical targets vs model outputs per class

**Question.** Per class, what are the analytical target means, what are the model's output means, and how closely do the second mirror the first?

**Why.** If the class ordering is already present in the targets, a model trained to reproduce those targets will reproduce the ordering whether or not it learned anything ecological. The question is how much of the model's per-class pattern is simply the target's.

**Data.** Targets: `targets_aux.npz` over all burned pixels and over burned *test* pixels. Model: `predictions_test.npz` (the only stored per-pixel model outputs; 15% test split, never seen in training). Severity, persistence and recovery gap are regression-head outputs. The model has no recurrence head: model recurrence is rebuilt with the Notebook 07b rule, `min(count of months with burned_prob > threshold / 5, 1)`, using the stored F1-optimised threshold and the per-month validity mask.

**Metric.** Class means with n. `small_n` flags n < 500 over all burned pixels, `small_n_test` over burned test pixels - the model columns rest on the test n. Table t03b gives, per indicator, the across-class Pearson and Spearman correlation of target vs model class means (classes with test n ≥ 500, and all classes), whether the class ordering is identical, and the pixel-level correlation for reference.

**Reading the output.** Across-class correlations use at most 10 points and are descriptive only.

In [ ]:
THR = float(preds["threshold"])
probs = preds["burned_probs"].reshape(len(TIDX), N_MONTHS)
pmask = preds["mask"].reshape(len(TIDX), N_MONTHS)
tb = BURNED[TIDX]
TB_IDX = TIDX[tb]
MODEL = {"severity": preds["severity"][tb].astype(np.float64),
         "persistence": preds["persistence"][tb].astype(np.float64),
         "recovery_gap": preds["recovery_gap"][tb].astype(np.float64),
         "recurrence": np.minimum(((probs > THR) & pmask).sum(axis=1) / 5.0, 1.0)[tb]}
TGT_TEST = {k: targets[k][TB_IDX].astype(np.float64) for k in IND}
lc_test = LC_ALL[TB_IDX]
SHORT = {"severity": "sev", "persistence": "per", "recovery_gap": "rg", "recurrence": "rec"}
note("burned test pixels: {:,}; stored threshold = {:.6f}".format(len(TB_IDX), THR))

rows = []
for c in LC_ORDER + ["all"]:
    ma = np.ones(N_BURNED, bool) if c == "all" else (lcb == c)
    mt = np.ones(len(TB_IDX), bool) if c == "all" else (lc_test == c)
    row = dict(land_cover="all burned" if c == "all" else LC_NAMES[c],
               n_burned=int(ma.sum()), n_burned_test=int(mt.sum()),
               small_n=bool(ma.sum() < MIN_N), small_n_test=bool(mt.sum() < MIN_N))
    for k in IND:
        s = SHORT[k]
        row[s + "_target_all"] = float(TG[k][ma].mean())
        row[s + "_target_test"] = float(TGT_TEST[k][mt].mean()) if mt.any() else np.nan
        row[s + "_model_test"] = float(MODEL[k][mt].mean()) if mt.any() else np.nan
    rows.append(row)
tvm = pd.DataFrame(rows)
tvm = pd.concat([tvm[tvm.land_cover != "all burned"].sort_values("rg_target_all", ascending=False),
                 tvm[tvm.land_cover == "all burned"]], ignore_index=True)
display(tvm)

cls = tvm[tvm.land_cover != "all burned"].set_index("land_cover")
big = cls[~cls.small_n_test]
crow = []
for k in IND:
    s = SHORT[k]
    for label, sub in (("classes with test n >= 500", big), ("all classes", cls)):
        xt, xm = sub[s + "_target_test"], sub[s + "_model_test"]
        crow.append(dict(
            indicator=k, class_set=label, k_classes=len(sub),
            pearson_class_means=float(stats.pearsonr(xt, xm)[0]),
            spearman_class_means=float(stats.spearmanr(xt, xm).statistic),
            ordering_identical=bool(list(xt.sort_values().index) == list(xm.sort_values().index)),
            max_abs_class_mean_diff=float((xm - xt).abs().max()),
            model_range_over_target_range=float((xm.max() - xm.min()) / (xt.max() - xt.min())),
            pearson_target_all_vs_target_test=float(stats.pearsonr(sub[s + "_target_all"], xt)[0]),
            pixel_pearson_burned_test=float(np.corrcoef(TGT_TEST[k], MODEL[k])[0, 1])))
tvm_corr = pd.DataFrame(crow)
display(tvm_corr)

SRC_M = {"*_target_all": "targets_aux.npz[severity|persistence|recovery_gap|recurrence], all burned pixels",
         "*_target_test": "targets_aux.npz[...] at predictions_test.npz['indices'], burned only",
         "sev/per/rg_model_test": "predictions_test.npz['severity'|'persistence'|'recovery_gap'], burned test pixels",
         "rec_model_test": "predictions_test.npz['burned_probs'] > predictions_test.npz['threshold'] & ['mask'], count/5 capped at 1",
         "land_cover": "X_scaled_meta.csv['lc_dominant']"}
save_table(tvm, "t03a_target_vs_model_by_class",
           "Analytical target means (all burned pixels and burned test pixels) beside model "
           "output means (burned test pixels) by land-cover class. small_n / small_n_test flag "
           "n < 500. Sorted by the recovery-gap target mean.", section=4, sources=SRC_M)
save_table(tvm_corr, "t03b_target_vs_model_class_mean_agreement",
           "Agreement between per-class target means and per-class model means on burned test "
           "pixels. 'model_range_over_target_range' < 1 means the model compresses the class "
           "spread. Descriptive: at most 10 classes.", section=4, sources=SRC_M)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.5, 9.5))
for ax, k in zip(axes.ravel(), IND):
    s = SHORT[k]
    for name, rrow in cls.iterrows():
        code = [cc for cc, nn in LC_NAMES.items() if nn == name][0]
        filled = not rrow.small_n_test
        ax.scatter(rrow[s + "_target_test"], rrow[s + "_model_test"], s=40 + 160 * np.sqrt(rrow.n_burned_test / cls.n_burned_test.max()),
                   facecolor=COLOR[code] if filled else "none", edgecolor=COLOR[code], lw=1.4, zorder=3)
        ax.annotate("{} ({:,})".format(name, int(rrow.n_burned_test)),
                    (rrow[s + "_target_test"], rrow[s + "_model_test"]),
                    fontsize=7, xytext=(4, 3), textcoords="offset points")
    lo = float(np.nanmin(cls[[s + "_target_test", s + "_model_test"]].to_numpy())) - 0.03
    hi = float(np.nanmax(cls[[s + "_target_test", s + "_model_test"]].to_numpy())) + 0.03
    ax.plot([lo, hi], [lo, hi], color="0.5", ls="--", lw=1)
    cr_ = tvm_corr[(tvm_corr.indicator == k) & (tvm_corr.class_set == "classes with test n >= 500")].iloc[0]
    ax.set_title("{}: r = {:.3f}, ρ = {:.3f} (k = {})".format(k, cr_.pearson_class_means,
                 cr_.spearman_class_means, cr_.k_classes), fontsize=10)
    ax.set_xlabel("target class mean (burned test pixels)")
    ax.set_ylabel("model class mean (burned test pixels)")
fig.suptitle("Per-class target vs model means — hollow markers: test n < 500 (excluded from r, ρ); "
             "label = test n", fontsize=10)
fig.tight_layout()
save_figure(fig, "f03_target_vs_model_class_means",
            "Per-class mean of each analytical target against the per-class mean of the "
            "corresponding model output, burned test pixels; dashed line is 1:1.",
            section=4, sources=SRC_M)
plt.show()

## 5. Recovery-gap target distribution per class

**Question.** In the target itself - before any model - how does the recovery gap differ between classes, and specifically, is cropland's gap lower than forest's, and by how much?

**Why.** The ordering has to be established on the target before asking whether the model, timing or seasonality explains it.

**Data.** `targets_aux.npz['recovery_gap']`, all burned pixels; `nbr_raw.npy` for the two window means that make up the target; `X_scaled_mask.npy` for the share of imputed observations inside each window.

**Metric.** Mean, median, 25th/75th percentiles, IQR, SD, and the share of pixels at each clip bound (`pct_at_0`: 2025 at or above 2017; `pct_at_1`: clipped at the ceiling). Saturation is reported beside every mean because a class mean near the ceiling is partly a count of clipped pixels. For cropland vs forest the target is also decomposed into its numerator (2017 mean − 2025 mean, in NBR units) and denominator (2017 mean).

**Reading the output.** Sorted by mean, highest gap first.

In [ ]:
rows = []
rg = TG["recovery_gap"]
for c in LC_ORDER:
    m = lcb == c
    n = int(m.sum())
    v = rg[m]
    vb = valid_b[m]
    rows.append(dict(land_cover=LC_NAMES[c], n=n, small_n=n < MIN_N,
                     mean=float(v.mean()), median=float(np.median(v)),
                     p25=float(np.percentile(v, 25)), p75=float(np.percentile(v, 75)),
                     iqr=float(np.percentile(v, 75) - np.percentile(v, 25)), sd=float(v.std()),
                     pct_at_0=100 * float((v <= EPS).mean()), pct_at_1=100 * float((v >= 1 - EPS).mean()),
                     mean_NBR_2017=float(b[m].mean()), mean_NBR_2025=float(r[m].mean()),
                     mean_drop_2017_minus_2025=float((b[m] - r[m]).mean()),
                     pct_imputed_obs_2017_window=100 * float(1 - vb[:, :12].mean()),
                     pct_imputed_obs_2025_window=100 * float(1 - vb[:, 96:].mean())))
rgdist = pd.DataFrame(rows).sort_values("mean", ascending=False)
display(rgdist)

fo, cr = lcb == 10, lcb == 40
crow = []
for metric, fn in [("n", lambda m: float(m.sum())),
                   ("recovery-gap target mean", lambda m: float(rg[m].mean())),
                   ("recovery-gap target median", lambda m: float(np.median(rg[m]))),
                   ("recovery-gap target p25", lambda m: float(np.percentile(rg[m], 25))),
                   ("recovery-gap target p75", lambda m: float(np.percentile(rg[m], 75))),
                   ("% at 0", lambda m: 100 * float((rg[m] <= EPS).mean())),
                   ("% clipped at 1", lambda m: 100 * float((rg[m] >= 1 - EPS).mean())),
                   ("numerator: mean NBR 2017 − mean NBR 2025", lambda m: float((b[m] - r[m]).mean())),
                   ("denominator: mean NBR 2017", lambda m: float(b[m].mean()))]:
    a, bb = fn(fo), fn(cr)
    crow.append(dict(metric=metric, forest=a, cropland=bb, cropland_minus_forest=bb - a,
                     cropland_over_forest=bb / a if a != 0 else np.nan))
rg_cf = pd.DataFrame(crow)
display(rg_cf)

SRC_R = {"recovery-gap statistics": "targets_aux.npz['recovery_gap'], burned pixels",
         "mean_NBR_2017 / mean_NBR_2025": "nbr_raw.npy[burned, 0:12] / [burned, 96:108] (as used by the target)",
         "pct_imputed_obs_*": "X_scaled_mask.npy[burned, 0:12] / [burned, 96:108]",
         "land_cover": "X_scaled_meta.csv['lc_dominant']"}
save_table(rgdist, "t04a_recovery_gap_distribution_by_class",
           "Recovery-gap target distribution by land-cover class, burned pixels, sorted by "
           "mean. pct_at_0 / pct_at_1 are the shares at the clip bounds.", section=5, sources=SRC_R)
save_table(rg_cf, "t04b_recovery_gap_cropland_vs_forest",
           "Cropland versus forest in the recovery-gap target, with the target decomposed into "
           "its numerator and denominator.", section=5, sources=SRC_R)

In [ ]:
order = rgdist[~rgdist.small_n]["land_cover"].tolist()
codes = {v: k for k, v in LC_NAMES.items()}
fig, ax = plt.subplots(figsize=(10, 4.6))
data = [rg[lcb == codes[nm]] for nm in order]
bp = ax.boxplot(data, whis=(5, 95), showfliers=False, showmeans=True, widths=0.6, patch_artist=True,
                meanprops=dict(marker="D", markerfacecolor="white", markeredgecolor="k", markersize=5),
                medianprops=dict(color="k"))
for patch, nm in zip(bp["boxes"], order):
    patch.set_facecolor(COLOR[codes[nm]])
    patch.set_alpha(0.55)
rd = rgdist.set_index("land_cover")
ax.set_xticks(range(1, len(order) + 1))
ax.set_xticklabels(["{}\nn={:,}\nat 1: {:.0f}%".format(nm, int(rd.loc[nm, "n"]), rd.loc[nm, "pct_at_1"])
                    for nm in order], fontsize=7.5)
ax.set_ylabel("recovery-gap target")
ax.set_title("Recovery-gap target by land cover, burned pixels (box = IQR, whiskers = 5–95th pct, "
             "◇ = mean); classes with n ≥ {}".format(MIN_N), fontsize=10)
fig.tight_layout()
save_figure(fig, "f04_recovery_gap_target_distribution",
            "Recovery-gap analytical target by land-cover class on burned pixels, sorted by mean. "
            "Box = IQR, whiskers = 5th-95th percentiles, diamond = mean; share clipped at 1 under "
            "each label.", section=5, sources=SRC_R)
plt.show()

## 6. Fixed-cohort control - first fire in 2018-2019

**Question.** Once recovery time is held (approximately) constant, does the class ordering of the recovery-gap target persist or collapse?

**Why.** This is the key check. If the ordering is a fire-timing artefact, restricting to pixels that first burned in the same two-year window - and so had comparable time to recover before 2025 - should shrink or scramble it.

**Data.** `targets_aux.npz['recovery_gap']`, `y_burned.npy`, `X_scaled_meta.csv`. Burned pixels.

**Metric.** Per class: recovery-gap target mean in (a) all burned pixels, (b) the cohort whose first fire is in 2018-2019, (c) the strict cohort, 2018-2019 *and* a single fire, so no later fire shortens recovery, and (d) a timing-standardised mean over all years - the class's mean within each first-fire year, weighted by the pooled first-fire-year distribution of all burned pixels (cells with n < 30 dropped; `std_weight_coverage` is the share of the standard weight retained). Mean months from first / last fire to Jan 2025 are shown to confirm the cohort equalises recovery time. Table t05b compares each population's class ordering with the all-burned ordering over the classes with n ≥ 500 in both: Spearman ρ, Kendall τ, whether the ordering is identical, the cropland − forest and grassland − forest differences, and the spread (max − min) of class means. Table t05c gives the class × first-fire-year means behind (d).

**Reading the output.** Persistence = ρ near 1 and a cropland − forest difference of similar size to the all-burned one. Collapse = ρ near 0 or the differences shrinking toward zero.

In [ ]:
coh = np.isin(ffy, [2018, 2019])
strict = coh & (n_fire == 1)
W_STD = pd.Series(ffy).value_counts(normalize=True).reindex(YEARS, fill_value=0.0)
MIN_CELL = 30
note("cohort (first fire 2018-2019): {:,} burned pixels; strict (and single fire): {:,}"
     .format(int(coh.sum()), int(strict.sum())))

rows, cell_rows = [], []
for c in LC_ORDER:
    m = lcb == c
    mc, ms = m & coh, m & strict
    num = den = 0.0
    for yr in YEARS:
        cell = m & (ffy == yr)
        k = int(cell.sum())
        if k > 0:
            cell_rows.append(dict(land_cover=LC_NAMES[c], first_fire_year=yr, n=k,
                                  rg_mean=float(rg[cell].mean()), rg_median=float(np.median(rg[cell])),
                                  pct_at_1=100 * float((rg[cell] >= 1 - EPS).mean()),
                                  mean_months_first_fire_to_2025=float((96 - first[cell]).mean()),
                                  below_min_cell=k < MIN_CELL))
        if k >= MIN_CELL:
            num += W_STD[yr] * float(rg[cell].mean())
            den += W_STD[yr]
    rows.append(dict(
        land_cover=LC_NAMES[c], n_all=int(m.sum()), rg_mean_all=float(rg[m].mean()),
        n_cohort=int(mc.sum()),
        rg_mean_cohort=float(rg[mc].mean()) if mc.any() else np.nan,
        rg_median_cohort=float(np.median(rg[mc])) if mc.any() else np.nan,
        pct_at_1_cohort=100 * float((rg[mc] >= 1 - EPS).mean()) if mc.any() else np.nan,
        cohort_months_first_fire_to_2025=float((96 - first[mc]).mean()) if mc.any() else np.nan,
        cohort_months_last_fire_to_2025=float((96 - last[mc]).mean()) if mc.any() else np.nan,
        n_strict=int(ms.sum()),
        rg_mean_strict=float(rg[ms].mean()) if ms.any() else np.nan,
        strict_months_fire_to_2025=float((96 - first[ms]).mean()) if ms.any() else np.nan,
        rg_mean_timing_standardised=num / den if den > 0 else np.nan,
        std_weight_coverage=den))
cohort_tab = pd.DataFrame(rows).sort_values("rg_mean_all", ascending=False)
cells = pd.DataFrame(cell_rows)


def _rank(s):
    return s.rank(ascending=False, method="min")


ct = cohort_tab.set_index("land_cover")
POPS = [("all burned", "rg_mean_all", ct.n_all >= MIN_N),
        ("first fire 2018-2019", "rg_mean_cohort", (ct.n_all >= MIN_N) & (ct.n_cohort >= MIN_N)),
        ("first fire 2018-2019, single fire", "rg_mean_strict", (ct.n_all >= MIN_N) & (ct.n_strict >= MIN_N)),
        ("timing-standardised, all years", "rg_mean_timing_standardised",
         (ct.n_all >= MIN_N) & (ct.std_weight_coverage >= 0.9))]
for label, col, elig in POPS:
    cohort_tab["rank_" + col.replace("rg_mean_", "")] = cohort_tab["land_cover"].map(_rank(ct.loc[elig, col]))
display(cohort_tab)

orow = []
for label, col, elig in POPS:
    sub = ct.loc[elig]
    ref = sub["rg_mean_all"]
    cur = sub[col]
    diff = lambda a, bb, s: float(s.loc[a] - s.loc[bb]) if (a in s.index and bb in s.index) else np.nan
    orow.append(dict(
        population=label, k_classes=int(elig.sum()), classes=", ".join(sub.index),
        spearman_vs_all_burned=float(stats.spearmanr(ref, cur).statistic),
        kendall_vs_all_burned=float(stats.kendalltau(ref, cur).statistic),
        ordering_identical=bool(list(ref.sort_values().index) == list(cur.sort_values().index)),
        cropland_minus_forest=diff("cropland", "forest", cur),
        cropland_minus_forest_all_burned=diff("cropland", "forest", ref),
        grassland_minus_forest=diff("grassland", "forest", cur),
        grassland_minus_forest_all_burned=diff("grassland", "forest", ref),
        spread_max_minus_min=float(cur.max() - cur.min()),
        spread_all_burned_same_classes=float(ref.max() - ref.min())))
cohort_order = pd.DataFrame(orow)
display(cohort_order)

SRC_C = {"recovery-gap means": "targets_aux.npz['recovery_gap'], burned pixels",
         "cohort / single-fire selection, months to 2025": "y_burned.npy (first / last labelled month, count)",
         "land_cover": "X_scaled_meta.csv['lc_dominant']"}
save_table(cohort_tab, "t05a_fixed_cohort_recovery_gap_by_class",
           "Recovery-gap target mean by land-cover class: all burned pixels; first fire in "
           "2018-2019; first fire in 2018-2019 with a single fire; and timing-standardised to "
           "the pooled first-fire-year distribution. Ranks (1 = highest gap) are among classes "
           "with n >= 500 in that population.", section=6, sources=SRC_C)
save_table(cohort_order, "t05b_fixed_cohort_ordering_comparison",
           "Does the land-cover ordering of the recovery-gap target survive holding recovery "
           "time constant? Each population compared with all burned pixels over the classes "
           "with n >= 500 in both.", section=6, sources=SRC_C)
save_table(cells, "t05c_recovery_gap_by_class_and_first_fire_year",
           "Recovery-gap target by land-cover class and year of first labelled fire, the cells "
           "behind the timing-standardised means (cells with n < 30 flagged and not used).",
           section=6, sources=SRC_C)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), gridspec_kw=dict(width_ratios=[1, 1.25]))
ax = axes[0]
plot_classes = ct[ct.n_all >= MIN_N].sort_values("rg_mean_all").index.tolist()
marks = [("rg_mean_all", "all burned", "o", "k"),
         ("rg_mean_cohort", "first fire 2018-19", "s", "tab:blue"),
         ("rg_mean_strict", "2018-19, single fire", "^", "tab:orange"),
         ("rg_mean_timing_standardised", "timing-standardized", "x", "tab:green")]
for j, (col, lab, mk, colr) in enumerate(marks):
    yy = np.arange(len(plot_classes)) + (j - 1.5) * 0.14
    ax.scatter(ct.loc[plot_classes, col], yy, marker=mk, color=colr, label=lab, zorder=3, s=30)
ax.set_yticks(range(len(plot_classes)))
ax.set_yticklabels(["{} (cohort n={:,})".format(k, int(ct.loc[k, "n_cohort"])) for k in plot_classes], fontsize=8)
ax.set_xlabel("recovery-gap target, class mean")
ax.set_title("Class means with recovery time held constant", fontsize=10)
ax.legend(fontsize=7.5, loc="lower right")

ax = axes[1]
for c in LC_ORDER:
    name = LC_NAMES[c]
    if ct.loc[name, "n_all"] < MIN_N:
        continue
    sub = cells[(cells.land_cover == name) & ~cells.below_min_cell]
    ax.plot(sub.first_fire_year, sub.rg_mean, marker="o", ms=3.5, lw=1.4, color=COLOR[c], label=name)
ax.axvspan(2017.5, 2019.5, color="0.9", zorder=0)
ax.set_xticks(YEARS)
ax.set_xlabel("year of first labeled fire (shaded: fixed cohort)")
ax.set_ylabel("recovery-gap target, class mean")
ax.set_title("Recovery-gap target by first-fire year (cells n ≥ {})".format(MIN_CELL), fontsize=10)
ax.legend(fontsize=7.5, ncol=3)
fig.tight_layout()
save_figure(fig, "f05_fixed_cohort_control",
            "Left: recovery-gap target class means for all burned pixels, the 2018-2019 "
            "first-fire cohort, the single-fire subset of that cohort, and timing-standardised "
            "means. Right: class means by year of first fire.", section=6, sources=SRC_C)
plt.show()

## 7. Baseline seasonality (2016-2017) per class

**Question.** How large is each class's NBR seasonal cycle in the 2016-2017 baseline, and does it track the class's recovery-gap target?

**Why.** The recovery gap divides by the 2017 annual mean NBR. Classes with a strong seasonal cycle around a low annual mean (herbaceous, rain-fed vegetation) have small, noisy denominators, so seasonality is a candidate channel through which the target could separate classes spectrally rather than ecologically.

**Data.** No 2016 raw observations are in the stored arrays. The 2016-2017 baseline enters them only through the z-score channel, `z = (NBR − μ_m) / σ_m`, with μ_m, σ_m the per-calendar-month baseline mean and population SD from Notebook 01. Because NBR and z are both stored for 9 years of the same calendar month, μ_m is recoverable exactly: for any two valid years a, b, σ = (x_a − x_b)/(z_a − z_b) and μ = x_a − z_a·σ. Sources: `nbr_raw.npy`, `nbr_zscore_raw.npy`, `X_scaled_mask.npy` (imputed months break the linear relation and are excluded). As a cross-check, the amplitude of the valid 2017 observations alone (the recovery gap's own reference window) is computed from `nbr_raw.npy`.

**Metric.** Reconstruction: μ_m is the median over pairs formed with the first valid year; a pixel-month is accepted when ≥ 2 pairs agree to within 1e-3. Table t06a verifies this per calendar month and uses z in 2017 to show which months actually contain 2016 data (n = 2 population SD ⇒ z₂₀₁₇ = ±1; z₂₀₁₇ = 0 ⇒ σ = 0, i.e. no 2016 composite, and the baseline is 2017 alone). Per pixel: seasonal amplitude = max − min of the 12 monthly baseline means (pixels with all 12 accepted). Per class: mean pixel amplitude, amplitude of the class-mean seasonal curve, that amplitude relative to the curve's annual mean, and the 2017-only amplitude. Correlations with the class recovery-gap target mean are across classes with n ≥ 500 (and without built-up and water); pixel-level Spearman is given for reference.

**Reading the output.** A class-level correlation over 7-9 points is descriptive only.

In [ ]:
ar = np.arange(N_BURNED)
MU = np.full((N_BURNED, 12), np.nan)
cal_rows = []
for cm in range(12):
    x = nbr_b[:, cm::12].astype(np.float64)          # (n, 9) years 2017..2025, this calendar month
    z = nbz_b[:, cm::12].astype(np.float64)
    v = valid_b[:, cm::12]
    refy = np.argmax(v, axis=1)                        # first valid year
    xr, zr = x[ar, refy], z[ar, refy]
    dz = z - zr[:, None]
    ok = v & v.any(axis=1)[:, None] & (np.abs(dz) > 0.5)
    sig = np.where(ok, (x - xr[:, None]) / np.where(ok, dz, 1.0), np.nan)
    mu = xr[:, None] - zr[:, None] * sig
    npair = ok.sum(axis=1)
    mu_med = np.nanmedian(mu, axis=1)
    spread = np.nanmax(mu, axis=1) - np.nanmin(mu, axis=1)
    accepted = (npair >= 2) & (spread < 1e-3)
    MU[accepted, cm] = mu_med[accepted]
    zz = z[v[:, 0], 0]
    p_pm1 = 100 * float((np.abs(np.abs(zz) - 1) < 0.02).mean())
    p_0 = 100 * float((np.abs(zz) < 0.02).mean())
    cal_rows.append(dict(
        calendar_month=cm + 1, pct_2017_obs_valid=100 * float(v[:, 0].mean()),
        pct_z2017_is_pm1=p_pm1, pct_z2017_is_0=p_0,
        baseline_composition=("2016 + 2017 (n = 2)" if p_pm1 > 90 else
                              "2017 only (no 2016 composite)" if p_0 > 90 else "mixed"),
        pct_pixel_months_reconstructed=100 * float(accepted.mean()),
        median_baseline_mean_NBR=float(np.nanmedian(mu_med[accepted])),
        median_baseline_sd=float(np.nanmedian(np.nanmedian(sig, axis=1)[accepted])),
        median_baseline_minus_2017_value=float(np.nanmedian((mu_med - x[:, 0])[accepted & v[:, 0]]))))
recon = pd.DataFrame(cal_rows)
display(recon)

full = np.isfinite(MU).all(axis=1)
amp1617 = np.where(full, np.nanmax(MU, axis=1) - np.nanmin(MU, axis=1), np.nan)
ann1617 = np.where(full, np.nanmean(MU, axis=1), np.nan)
w17 = np.where(valid_b[:, :12], nbr_b[:, :12], np.nan)
ok17 = valid_b[:, :12].sum(axis=1) >= 10
amp2017 = np.where(ok17, np.nanmax(w17, axis=1) - np.nanmin(w17, axis=1), np.nan)
note("burned pixels with all 12 baseline months reconstructed: {:,} ({:.1f}%)"
     .format(int(full.sum()), 100 * full.mean()))

rows, BCURVE = [], {}
for c in LC_ORDER:
    m = lcb == c
    mf = m & full
    n = int(m.sum())
    curve = MU[mf].mean(axis=0) if mf.any() else np.full(12, np.nan)
    BCURVE[c] = curve
    rows.append(dict(
        land_cover=LC_NAMES[c], n=n, small_n=n < MIN_N,
        n_full_baseline=int(mf.sum()), pct_full_baseline=100 * mf.sum() / n,
        baseline_annual_mean_NBR=float(ann1617[mf].mean()),
        ref_2017_mean_NBR=float(b[m].mean()),
        amplitude_pixel_mean=float(amp1617[mf].mean()),
        amplitude_pixel_median=float(np.median(amp1617[mf])),
        amplitude_class_curve=float(curve.max() - curve.min()),
        relative_amplitude_class_curve=float((curve.max() - curve.min()) / abs(curve.mean())),
        peak_month=int(np.argmax(curve) + 1), trough_month=int(np.argmin(curve) + 1),
        amplitude_2017_only_pixel_mean=float(np.nanmean(amp2017[m])),
        rg_target_mean=float(rg[m].mean()),
        rg_target_mean_full_baseline_pixels=float(rg[mf].mean())))
season = pd.DataFrame(rows).sort_values("rg_target_mean", ascending=False)
display(season)

sn = season.set_index("land_cover")
SETS = [("classes with n >= 500", sn[~sn.small_n]),
        ("n >= 500, excluding built-up and water", sn[~sn.small_n & ~sn.index.isin(["built-up", "water"])])]
crow = []
for var in ["amplitude_pixel_mean", "amplitude_class_curve", "relative_amplitude_class_curve",
            "amplitude_2017_only_pixel_mean", "baseline_annual_mean_NBR", "ref_2017_mean_NBR"]:
    for label, sub in SETS:
        crow.append(dict(variable=var, class_set=label, k_classes=len(sub),
                         pearson_vs_rg_target_mean=float(stats.pearsonr(sub[var], sub["rg_target_mean"])[0]),
                         spearman_vs_rg_target_mean=float(stats.spearmanr(sub[var], sub["rg_target_mean"]).statistic)))
# pixel level, for reference
pm = full & np.isin(lcb, [c for c in LC_ORDER if (lcb == c).sum() >= MIN_N])
within = []
for c in LC_ORDER:
    mc = pm & (lcb == c)
    if mc.sum() >= MIN_N:
        within.append((mc.sum(), stats.spearmanr(amp1617[mc], rg[mc]).statistic))
crow.append(dict(variable="amplitude (pixel) vs recovery gap (pixel), pooled", class_set="burned pixels, classes n >= 500",
                 k_classes=int(pm.sum()), pearson_vs_rg_target_mean=np.nan,
                 spearman_vs_rg_target_mean=float(stats.spearmanr(amp1617[pm], rg[pm]).statistic)))
crow.append(dict(variable="amplitude (pixel) vs recovery gap (pixel), within-class mean (n-weighted)",
                 class_set="burned pixels, classes n >= 500", k_classes=int(pm.sum()),
                 pearson_vs_rg_target_mean=np.nan,
                 spearman_vs_rg_target_mean=float(np.average([w for _, w in within], weights=[k for k, _ in within]))))
season_corr = pd.DataFrame(crow)
display(season_corr)

SRC_S = {"baseline monthly means (2016-2017)": "reconstructed from nbr_raw.npy and nbr_zscore_raw.npy (valid months, X_scaled_mask.npy)",
         "amplitude_2017_only_pixel_mean": "nbr_raw.npy[burned, 0:12], valid months only",
         "ref_2017_mean_NBR": "nbr_raw.npy[burned, 0:12] (the recovery-gap denominator)",
         "rg_target_mean": "targets_aux.npz['recovery_gap']",
         "land_cover": "X_scaled_meta.csv['lc_dominant']"}
save_table(recon, "t06a_baseline_reconstruction_by_calendar_month",
           "Verification of the 2016-2017 per-calendar-month baseline mean reconstructed from "
           "the stored NBR and NBR z-score arrays, burned pixels. z in 2017 identifies the "
           "baseline's composition: ±1 for a two-year (2016 + 2017) population SD, 0 where no "
           "2016 composite contributed.", section=7, sources=SRC_S)
save_table(season, "t06b_baseline_seasonality_by_class",
           "Baseline NBR seasonality by land-cover class (burned pixels): amplitude = max − min "
           "of the 12 monthly 2016-2017 baseline means; relative amplitude divides the class "
           "curve's amplitude by its annual mean.", section=7, sources=SRC_S)
save_table(season_corr, "t06c_seasonality_vs_recovery_gap_correlation",
           "Correlation of baseline seasonality (and baseline level) with the recovery-gap "
           "target, across land-cover classes and, for reference, across pixels.",
           section=7, sources=SRC_S)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for c in LC_ORDER:
    name = LC_NAMES[c]
    if sn.loc[name, "small_n"]:
        continue
    axes[0].plot(range(1, 13), BCURVE[c], marker="o", ms=3, lw=1.5, color=COLOR[c],
                 label="{} (n={:,})".format(name, int(sn.loc[name, "n_full_baseline"])))
for cm in recon.loc[recon.baseline_composition != "2016 + 2017 (n = 2)", "calendar_month"]:
    axes[0].axvspan(cm - 0.5, cm + 0.5, color="0.9", zorder=0)
axes[0].set_xticks(range(1, 13))
axes[0].set_xlabel("calendar month (shaded: baseline is 2017 only)")
axes[0].set_ylabel("baseline mean NBR (2016–2017)")
axes[0].set_title("Baseline seasonal NBR cycle by class", fontsize=10)
axes[0].legend(fontsize=7, ncol=2)
for ax, var, xl in ((axes[1], "amplitude_class_curve", "seasonal amplitude of class baseline curve (NBR)"),
                    (axes[2], "relative_amplitude_class_curve", "amplitude / annual baseline mean")):
    sub = sn[~sn.small_n]
    for name, rrow in sub.iterrows():
        code = [cc for cc, nn in LC_NAMES.items() if nn == name][0]
        ax.scatter(rrow[var], rrow.rg_target_mean, color=COLOR[code], s=45, zorder=3)
        ax.annotate(name, (rrow[var], rrow.rg_target_mean), fontsize=7, xytext=(4, 3), textcoords="offset points")
    cc_ = season_corr[(season_corr.variable == var) & (season_corr.class_set == "classes with n >= 500")].iloc[0]
    ax.set_title("r = {:.2f}, ρ = {:.2f} (k = {})".format(cc_.pearson_vs_rg_target_mean,
                 cc_.spearman_vs_rg_target_mean, cc_.k_classes), fontsize=10)
    ax.set_xlabel(xl)
    ax.set_ylabel("recovery-gap target, class mean")
fig.tight_layout()
save_figure(fig, "f06_baseline_seasonality",
            "Left: 2016-2017 baseline mean NBR by calendar month per class (reconstructed from the "
            "stored NBR and z-score arrays; shaded months have no 2016 composite). Middle/right: "
            "class seasonal amplitude, absolute and relative to the annual baseline mean, against "
            "the class recovery-gap target mean.", section=7, sources=SRC_S)
plt.show()

## 8. Verdict

**What this cell does.** Pulls every number in the verdict from the tables above - nothing is typed in by hand - and asserts each qualitative claim before stating it, so if the data change the notebook stops rather than printing a claim the numbers no longer support. The text is written to `diagnostics/nb13_landcover_timing/nb13_verdict.md` with the table each number comes from in brackets.

**Questions answered.** (i) Is the per-class ordering already in the target before any model? (ii) Is it explained by fire timing (§2 vs §6)? (iii) Does baseline seasonality contribute (§7)? (iv) How much does the model just mirror the target (§4)? (v) Should the result be described as spectral rather than ecological?

In [ ]:
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


def ym(d):
    y = int(np.floor(d + 1e-9))
    return "{} {}".format(MONTHS[int(round((d - y) * 12))], y)


rd = rgdist.set_index("land_cover")
tcs = timing.set_index("land_cover")
tcf = timing_cf.set_index("metric")
es = event_summary.set_index("land_cover")
co = cohort_order.set_index("population")
ctab = cohort_tab.set_index("land_cover")
ag = tvm_corr.set_index(["indicator", "class_set"])
sc = season_corr.set_index(["variable", "class_set"])
BIG, S9, S7 = "classes with test n >= 500", "classes with n >= 500", "n >= 500, excluding built-up and water"
coh_classes = co.loc["first fire 2018-2019", "classes"].split(", ")
es_big = es[es.n_fires_2018_2019 >= MIN_N]

V = dict(
    rg_crop=rd.loc["cropland", "mean"], rg_for=rd.loc["forest", "mean"],
    med_crop=rd.loc["cropland", "median"], med_for=rd.loc["forest", "median"],
    num_crop=rd.loc["cropland", "mean_drop_2017_minus_2025"], num_for=rd.loc["forest", "mean_drop_2017_minus_2025"],
    den_crop=rd.loc["cropland", "mean_NBR_2017"], den_for=rd.loc["forest", "mean_NBR_2017"],
    ff_crop=tcs.loc["cropland", "median_first_fire"], ff_for=tcs.loc["forest", "median_first_fire"],
    mo_crop=tcs.loc["cropland", "mean_months_first_fire_to_2025_window"],
    mo_for=tcs.loc["forest", "mean_months_first_fire_to_2025_window"],
    ks=tcf.loc["KS statistic D (first-fire date distributions)", "cropland_minus_forest"],
    coh_mo_lo=ctab.loc[coh_classes, "cohort_months_first_fire_to_2025"].min(),
    coh_mo_hi=ctab.loc[coh_classes, "cohort_months_first_fire_to_2025"].max(),
    k_coh=int(co.loc["first fire 2018-2019", "k_classes"]),
    rho_coh=co.loc["first fire 2018-2019", "spearman_vs_all_burned"],
    rho_strict=co.loc["first fire 2018-2019, single fire", "spearman_vs_all_burned"],
    rho_std=co.loc["timing-standardised, all years", "spearman_vs_all_burned"],
    cf_all=co.loc["all burned", "cropland_minus_forest"],
    cf_coh=co.loc["first fire 2018-2019", "cropland_minus_forest"],
    cf_std=co.loc["timing-standardised, all years", "cropland_minus_forest"],
    coh_lo=ctab.loc[coh_classes, "rg_mean_cohort"].min(), coh_hi=ctab.loc[coh_classes, "rg_mean_cohort"].max(),
    amp_r9=sc.loc[("amplitude_class_curve", S9), "pearson_vs_rg_target_mean"],
    k9=int(sc.loc[("amplitude_class_curve", S9), "k_classes"]),
    within=sc.loc[("amplitude (pixel) vs recovery gap (pixel), within-class mean (n-weighted)",
                   "burned pixels, classes n >= 500"), "spearman_vs_rg_target_mean"],
    rel_r7=sc.loc[("relative_amplitude_class_curve", S7), "pearson_vs_rg_target_mean"],
    k7=int(sc.loc[("relative_amplitude_class_curve", S7), "k_classes"]),
    r_rg=ag.loc[("recovery_gap", BIG), "pearson_class_means"], rho_rg=ag.loc[("recovery_gap", BIG), "spearman_class_means"],
    same_rg=bool(ag.loc[("recovery_gap", BIG), "ordering_identical"]),
    maxd_rg=ag.loc[("recovery_gap", BIG), "max_abs_class_mean_diff"], k_t=int(ag.loc[("recovery_gap", BIG), "k_classes"]),
    r_sev=ag.loc[("severity", BIG), "pearson_class_means"], r_per=ag.loc[("persistence", BIG), "pearson_class_means"],
    r_rec=ag.loc[("recurrence", BIG), "pearson_class_means"],
    y2_lo=es_big["dNBR_year2_mean_fires_2018_2019"].min(), y2_hi=es_big["dNBR_year2_mean_fires_2018_2019"].max(),
    y2_for_early=es.loc["forest", "dNBR_year2_mean_fires_2018_2019"],
    y2_crop_early=es.loc["cropland", "dNBR_year2_mean_fires_2018_2019"],
    y2_grass_early=es.loc["grassland", "dNBR_year2_mean_fires_2018_2019"],
    y2_for_all=es.loc["forest", "dNBR_year2_mean"], ff_for_event=es.loc["forest", "median_fire_date"],
)

# ---- guards: each qualitative claim below must be supported by the numbers ----
assert V["rg_crop"] > V["rg_for"], "claim: cropland gap exceeds forest gap in the target"
assert 0.8 < V["num_crop"] / V["num_for"] < 1.5 and V["den_crop"] / V["den_for"] < 0.6, "claim: denominator, not numerator"
assert V["ff_for"] > V["ff_crop"] and V["mo_for"] < V["mo_crop"], "claim: forest burned later than cropland"
assert min(V["rho_coh"], V["rho_strict"], V["rho_std"]) >= 0.9, "claim: ordering persists with timing controlled"
assert 0 < V["cf_coh"] < V["cf_all"] and 0 < V["cf_std"] < V["cf_all"], "claim: cropland-forest narrows but persists"
assert abs(V["amp_r9"]) < 0.3 and abs(V["within"]) < 0.2 and V["rel_r7"] > 0.7, "claim: only relative amplitude tracks"
assert V["r_rg"] > 0.99 and V["same_rg"] and V["r_sev"] > 0.99 and V["r_per"] > 0.99, "claim: model mirrors target"
assert es_big["dNBR_year2_mean_fires_2018_2019"].abs().max() < 0.05, "claim: all classes back near pre-fire by year 2"
assert V["y2_for_all"] < -0.05 and abs(V["y2_for_early"]) < 0.02, "claim: forest depression is specific to later fires"

s1 = ("**(i) The ordering is in the target before any model.** In the recovery-gap target cropland scores "
      "{rg_crop:.3f} and forest {rg_for:.3f} (difference +{d:.3f}; medians {med_crop:.3f} vs {med_for:.3f}) — cropland is "
      "higher, not lower — and the difference comes from the denominator rather than from recovery: the two "
      "classes lost similar absolute NBR between 2017 and 2025 ({num_crop:.3f} vs {num_for:.3f}), but cropland's "
      "2017 reference level is half of forest's ({den_crop:.3f} vs {den_for:.3f}) [t04a, t04b]."
      ).format(d=V["rg_crop"] - V["rg_for"], **V)
s2 = ("**(ii) Fire timing does not explain it.** The classes did burn at different times — median first fire "
      "{ffc} for cropland vs {fff} for forest, {mo_crop:.0f} vs {mo_for:.0f} months before the 2025 window, KS D = "
      "{ks:.2f} [t01b, t01c] — but in the wrong direction (forest had less time to recover yet has the lower gap), "
      "and with recovery time held at {coh_mo_lo:.0f}-{coh_mo_hi:.0f} months in the 2018-2019 cohort the ordering "
      "persists (Spearman ρ = {rho_coh:.2f} over {k_coh} classes; {rho_strict:.2f} for single-fire pixels; "
      "{rho_std:.2f} timing-standardised) while cropland − forest narrows from {cf_all:.2f} to {cf_coh:.2f}-{cf_std:.2f} "
      "[t05a, t05b]: timing inflates that pairwise difference but does not create the ordering."
      ).format(ffc=ym(V["ff_crop"]), fff=ym(V["ff_for"]), **V)
s3 = ("**(iii) Baseline seasonality contributes only through the same denominator.** The absolute 2016-2017 "
      "seasonal amplitude does not track the target (r = {amp_r9:.2f} across {k9} classes; within-class pixel "
      "Spearman ρ = {within:.2f}); amplitude correlates only once divided by the annual baseline mean "
      "(r = {rel_r7:.2f}, {k7} classes, descriptive), which is the low-baseline normalisation effect again rather "
      "than an independent contribution [t06b, t06c]."
      ).format(**V)
s4 = ("**(iv) The model mirrors the target, and the target does not measure recovery.** Per-class model means "
      "reproduce the target almost exactly (recovery gap r = {r_rg:.4f}, ρ = {rho_rg:.2f}, identical ordering over "
      "{k_t} classes, largest class-mean difference {maxd_rg:.3f}; severity r = {r_sev:.3f}; persistence "
      "r = {r_per:.4f}; only the threshold-derived recurrence departs, r = {r_rec:.2f}) [t03a, t03b], whereas NBR "
      "measured on each pixel's own fire shows no class ordering once calendar period is held fixed — for 2018-2019 "
      "single fires every class with n ≥ 500 is back within {y2_lo:+.3f} to {y2_hi:+.3f} NBR of its pre-fire level "
      "by year 2 (forest {y2_for_early:+.3f}, cropland {y2_crop_early:+.3f}, grassland {y2_grass_early:+.3f}; forest's "
      "apparent {y2_for_all:+.3f} depression belongs to its {fffe}-centred fires) [t02b] while the same cohort's "
      "target spans {coh_lo:.2f}-{coh_hi:.2f} [t05a] — so the result should be described as the model reproducing "
      "spectral, baseline-normalised NBR patterns built into its target, not ecological recovery differences "
      "between land covers."
      ).format(fffe=ym(V["ff_for_event"]), **V)

verdict = "\n\n".join([s1, s2, s3, s4])
from IPython.display import Markdown
display(Markdown(verdict))
vpath = os.path.join(OUT, PREFIX + "verdict.md")
with open(vpath, "w") as fh:
    fh.write("<!-- generated by notebooks/13_landcover_timing_diagnostic.ipynb §8 on {} -->\n\n".format(RUN_STAMP))
    fh.write("# Verdict — land-cover recovery-gap diagnostic\n\n" + verdict + "\n\n")
    fh.write("Table ids in brackets refer to `{}/tables/nb13_<id>_*.csv`.\n".format(OUT_REL))
REGISTRY.append(dict(section="8", kind="verdict", stem=PREFIX + "verdict", files=os.path.join(OUT_REL, PREFIX + "verdict.md"),
                     caption="Verdict, every number pulled from the §2-§7 tables.",
                     sources="nb13 tables t01b, t01c, t02b, t03b, t04a, t04b, t05a, t05b, t06b, t06c"))
print("  ->", os.path.relpath(vpath, ROOT))

## 9. Result registry

Every table and figure this notebook wrote, the section that produced it, and the arrays it is computed from.

In [ ]:
REGISTRY.append(dict(section="9", kind="table", stem=PREFIX + "t99_result_registry",
                     files=os.path.join(OUT_REL, "tables", PREFIX + "t99_result_registry") + ".csv, .md",
                     caption="This registry.", sources="REGISTRY"))
registry = pd.DataFrame(REGISTRY)[["section", "kind", "stem", "files", "sources", "caption"]]
registry.to_csv(os.path.join(TAB, PREFIX + "t99_result_registry.csv"), index=False)
with open(os.path.join(TAB, PREFIX + "t99_result_registry.md"), "w") as fh:
    fh.write("# Result registry — notebooks/13_landcover_timing_diagnostic.ipynb\n\nGenerated {}.\n\n".format(RUN_STAMP))
    fh.write(_markdown_table(registry.drop(columns=["caption"])))
display(registry.drop(columns=["caption"]))
written = sorted(os.listdir(TAB) + os.listdir(FIG))
print(len(written), "files under", OUT_REL)